# 🚀 Pipeline ALPR Ligero Orientado a Eventos (Vía de Lastre - Pintag)
**Arquitectura:** Crop Físico + Motion Gate 3 Estados + YOLOv8 + ByteTrack + Top-M/Top-K + FastPlateOCR + Deduplicación + Reporte Excel con Fotos Incrustadas.

> **Aceleración GPU:** Asegúrate de que el entorno de ejecución tenga una GPU asignada:
>  →  → .

In [ ]:
# 1. Verificar GPU NVIDIA disponible en Colab
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Instalar dependencias del proyecto
!pip install -q ultralytics fast_alpr supervision openpyxl opencv-python-headless pyyaml onnxruntime-gpu
print("✅ Dependencias instaladas con soporte GPU.")

In [ ]:
# 3. Clonar repositorio o verificar carpeta
import os, glob

if not os.path.exists("main.py"):
    !git clone https://github.com/riofutabac/PlacasVideos.git
    %cd PlacasVideos
    print("✅ Repositorio clonado y activo.")
else:
    print(f"✅ Directorio actual del proyecto: {os.getcwd()}")

# Verificar videos disponibles
videos = glob.glob("*.mp4")
print(f"📹 Videos .mp4 en directorio: {videos}")
if not videos:
    print("⚠️ Recuerda subir tus videos .mp4 a esta carpeta o conectarlos desde Google Drive!")


In [ ]:
# 4. Ejecutar el Pipeline ALPR de Extremo a Extremo en GPU
!python main.py

In [ ]:
# 5. Visualizar la Telemetría Almacenada en SQLite
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/events.sqlite")
print("=== ÚLTIMA CORRIDA ===")
display(pd.read_sql("SELECT run_id, total_videos, total_events, speed_ratio, started_at, finished_at FROM processing_runs ORDER BY rowid DESC LIMIT 1", conn))

print("
=== TELEMETRÍA POR CLIP ===")
display(pd.read_sql("SELECT clip_id, wall_clock_seconds, speed_ratio, decode_seconds, vehicle_detection_seconds, processed_events FROM processed_clips ORDER BY rowid DESC LIMIT 2", conn))

print("
=== EVENTOS DETECTADOS ===")
display(pd.read_sql("SELECT event_id, datetime_str, direction, vehicle_type, plate_raw, plate_corrected, plate_status, duplicate_of FROM events ORDER BY rowid DESC LIMIT 10", conn))
conn.close()

In [ ]:
# 6. Galería de Evidencias (Fotos de Vehículos y Recortes de Placa)
import glob
from IPython.display import Image, display

veh_images = sorted(glob.glob("evidence/vehicles/*.jpg"))
print(f"Total fotos de vehículos guardadas: {len(veh_images)}")
for img_path in veh_images[:4]:
    print(f"Evidencia: {img_path}")
    display(Image(filename=img_path, width=400))

In [ ]:
# 7. Descargar el Reporte Excel de Auditoría
try:
    from google.colab import files
    if os.path.exists("reports/reporte_auditoria.xlsx"):
        files.download("reports/reporte_auditoria.xlsx")
        print("📥 Descargando reporte_auditoria.xlsx...")
except Exception as e:
    print(f"Descarga manual disponible en reports/reporte_auditoria.xlsx: {e}")